In [ ]:
import h5py
import os
import obspy
import numpy as np
import pandas as pd
from tqdm import tqdm

# --- KONFIGURASI ---
BASE_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/output_waveform_indonesia_0109'
CSV_METADATA = os.path.join(BASE_DIR, 'Master_Metadata_Generated.csv')
OUTPUT_H5 = os.path.join(BASE_DIR, 'STEAD_Indonesia_Final_2004_2010.hdf5')

# 1. Load Metadata Master
print("Memuat Metadata Master...")
df_master = pd.read_csv(CSV_METADATA)
df_master.set_index('trace_name', inplace=True)

def pad_or_trim(data, n=6000):
    if len(data) >= n: return data[:n]
    else: return np.pad(data, (0, n - len(data)), mode='constant')

print("Memulai pembuatan Brankas HDF5 Final...")

# 2. Proses Merger
with h5py.File(OUTPUT_H5, 'w') as hf:
    for _, row in tqdm(df_master.iterrows(), total=len(df_master)):
        file_path = row['source_file']
        trace_name = row.name # Mengambil trace_name dari index
        
        try:
            st = obspy.read(file_path, headonly=True)
            if len(st) >= 3:
                # Membaca data penuh setelah validasi header
                st = obspy.read(file_path)
                data_list = [pad_or_trim(tr.data) for tr in st[:3]]
                data_array = np.column_stack(data_list)
                
                # Simpan dataset
                dset = hf.create_dataset(trace_name, data=data_array)
                
                # INJEKSI METADATA dari CSV Master ke dalam dset.attrs
                for col in df_master.columns:
                    val = row[col]
                    dset.attrs[col] = str(val) if pd.isna(val) else val
                    
        except Exception as e:
            continue

print(f"Selesai! Brankas HDF5 Final tersimpan di: {OUTPUT_H5}")